# 03 · RQ1 – Location of GenAI exposure in the Record-to-Report process (Section 4.1)

Produces Figures 1–10 in the order of Section 4.1, plus the Release 4 radar chart for Appendix D (Figure D1). Requires `01_constructs_and_checks.ipynb`.

File names start with the figure number; where a figure has two panels, `a` is the **left** panel and `b` the **right** panel as placed in the thesis.

| Thesis item | Output in `outputs/2_ch4_1_rq1_location/` |
|---|---|
| Figure 1 – usage share by subprocess and platform, R1–R5 | `fig01_usage_share_by_subprocess_platform.png` (+ `fig01_data.csv`) |
| Figure 2 – automation vs augmentation, whole process, by platform | `fig02_collaboration_mode_by_platform.png` (+ `fig02_04_data.csv`) |
| Figure 3 – automation vs augmentation by subprocess, Claude.ai | `fig03_collaboration_mode_by_subprocess_claude_ai.png` |
| Figure 4 – automation vs augmentation by subprocess, 1P API | `fig04_collaboration_mode_by_subprocess_1p_api.png` |
| Figure 5 – automation overrepresentation index | `fig05a_automation_index_claude_ai.png`, `fig05b_automation_index_1p_api.png` |
| Figure 6 – theoretical capability vs observed exposure, R5 | `fig06a_radar_R5_gateON.png`, `fig06b_radar_R5_gateOFF.png` |
| Figure 7 – subprocess concentration vs exposure, R5 | `fig07a_concentration_vs_exposure_R5_gateON.png`, `fig07b_..._gateOFF.png` |
| Figure 8 – concentration R4 → R5 | `fig08a_concentration_trend_gateON.png`, `fig08b_concentration_trend_gateOFF.png` |
| Figure 9 – task dispersion vs subprocess exposure, R5 | `fig09a_task_dispersion_vs_exposure_R5_gateON.png`, `fig09b_..._gateOFF.png` |
| Figure 10 – Lorenz curves R5 and Gini trend | `fig10a_lorenz_R5.png`, `fig10b_gini_trend.png` (+ `fig10_gini_top20.csv`) |
| Figure D1 – radar chart, R4 (Appendix D) | `outputs/4_appendix_D/figD1a_radar_R4_gateON.png`, `figD1b_radar_R4_gateOFF.png` |
| Slopes / R² quoted in the text | `fit_statistics.csv` |

In [1]:
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import r2r_common as rc

pd.set_option("display.width", 160)
tm, panel = rc.load_data()
sub_exp = rc.read_construct("subprocess_exposure.csv")
proc_exp = rc.read_construct("process_exposure.csv")
task_disp = rc.read_construct("task_dispersion.csv")
sub_conc = rc.read_construct("subprocess_concentration.csv")
proc_conc = rc.read_construct("process_concentration.csv")
gate_tag = lambda g: "gateON" if g else "gateOFF"
gate_txt = lambda g: "gate ON" if g else "gate OFF"

[load] task_master: 212 rows, 91 in scope ({'Master Data Management': 8, 'Record': 31, 'Close & Consolidation': 5, 'Report': 13, 'Analysis & Business Insight': 12, 'Planning & Budgeting': 22})
[load] panel: 9,522 rows, releases [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)], 50 tasks; rows whose task is not in scope: 0


## 4.1.1 Platform usage and collaboration mode

### Figure 1 – Task usage share by subprocess and platform, Releases 1–5
Left panel: each platform's share of all R2R usage (raw AEI pct summed over in-scope tasks, divided by the two platforms' combined total). Right panels: each platform's own R2R usage (renormalised to 100%) split across subprocesses.

In [2]:
def platform_subprocess_share(releases=rc.ANALYSIS_RELEASES):
    rows = []
    sp_map = tm.set_index("task_id")["subprocess"]
    for r in releases:
        raw = {}
        for p in rc.RELEASE_PLATFORMS[r]:
            share = rc.get_r2r_task_pct(panel, r, p)
            by_sp = share.groupby(share.index.map(sp_map)).sum().reindex(rc.SUBPROCESS_ORDER).fillna(0) * 100
            for sp, v in by_sp.items():
                rows.append({"release": r, "platform": p, "subprocess": sp, "share_pct": v})
            m = (panel["release"] == r) & (panel["dimension_n"] == "usage") & (panel["variable_n"] == "pct") & (panel["platform_n"] == p)
            raw[p] = panel.loc[m, "value"].sum()
        total = sum(raw.values())
        for p, v in raw.items():
            rows.append({"release": r, "platform": p, "subprocess": "Entire Process", "share_pct": 100 * v / total if total else np.nan})
    return pd.DataFrame(rows)


fig01 = platform_subprocess_share()
rc.save_csv(fig01, "rq1", "fig01_data.csv")

releases = list(rc.ANALYSIS_RELEASES)
mosaic = [["ALL", "ALL", "MDM", "REC", "CNC"], ["ALL", "ALL", "REP", "ABI", "PNB"]]
sp_key = {"MDM": "Master Data Management", "REC": "Record", "CNC": "Close & Consolidation",
          "REP": "Report", "ABI": "Analysis & Business Insight", "PNB": "Planning & Budgeting"}
fig, axd = plt.subplot_mosaic(mosaic, figsize=(17, 7.5))
for p in ["claude_ai", "1p_api"]:
    s = fig01[(fig01["subprocess"] == "Entire Process") & (fig01["platform"] == p)].sort_values("release")
    axd["ALL"].plot(s["release"], s["share_pct"], color=rc.PLATFORM_COLORS[p], marker=rc.PLATFORM_MARKERS[p],
                    linewidth=2.6, markersize=6.5, markeredgecolor="white", markeredgewidth=0.8)
axd["ALL"].set_title("Entire Process", fontsize=13)
axd["ALL"].set_ylabel("Share of R2R usage (%)", fontsize=11)
rc.capped_ylim(axd["ALL"], 100)
rc.release_ticks(axd["ALL"], releases)
axd["ALL"].tick_params(labelsize=9.5)
rc.clean_axes(axd["ALL"])
sub_only = fig01[fig01["subprocess"] != "Entire Process"]
shared_ylim = (0, sub_only["share_pct"].max() * 1.08)
for key, sp in sp_key.items():
    ax = axd[key]
    for p in ["claude_ai", "1p_api"]:
        s = sub_only[(sub_only["subprocess"] == sp) & (sub_only["platform"] == p)].sort_values("release")
        ax.plot(s["release"], s["share_pct"], color=rc.PLATFORM_COLORS[p], marker=rc.PLATFORM_MARKERS[p],
                linewidth=1.8, markersize=4.5, markeredgecolor="white", markeredgewidth=0.6)
    ax.set_title(sp, fontsize=9.5)
    rc.release_ticks(ax, releases, short=True)
    ax.tick_params(labelsize=7.5)
    ax.set_ylabel("Usage share (%)", fontsize=8, labelpad=6)
    ax.set_ylim(*shared_ylim)
    rc.clean_axes(ax)
rc.legend_box(fig, [(rc.PLATFORM_COLORS[p], rc.PLATFORM_LABELS[p], rc.PLATFORM_MARKERS[p]) for p in ["claude_ai", "1p_api"]],
              loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=2)
fig.suptitle("Task usage share trends by subprocess, by platform (R1-R5)", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0.04, 1, 0.94])
rc.save_figure(fig, "rq1", "fig01_usage_share_by_subprocess_platform.png")
fig01.pivot_table(index=["subprocess", "platform"], columns="release", values="share_pct").round(1)

[write] outputs/2_ch4_1_rq1_location/fig01_data.csv


[write] outputs/2_ch4_1_rq1_location/fig01_usage_share_by_subprocess_platform.png


release                                    1      2     3     4     5
subprocess                  platform                                 
Analysis & Business Insight 1p_api       NaN    NaN  57.8  39.8  41.1
                            claude_ai   34.4   28.6  56.3  51.9  48.8
Close & Consolidation       1p_api       NaN    NaN   1.7   4.8   3.2
                            claude_ai    1.5    1.3   1.8   2.6   2.9
Entire Process              1p_api       NaN    NaN  52.8  42.0  39.8
                            claude_ai  100.0  100.0  47.2  58.0  60.2
Master Data Management      1p_api       NaN    NaN  10.1  13.0  12.7
                            claude_ai   26.7   27.5   8.2   8.4   9.0
Planning & Budgeting        1p_api       NaN    NaN   4.1   4.5   2.9
                            claude_ai   21.5   20.5  14.3  13.0  12.9
Record                      1p_api       NaN    NaN  23.4  31.8  34.7
                            claude_ai   12.8   17.2  14.8  19.3  20.5
Report                      1p_api       NaN    NaN   2.9   6.1   5.4
                            claude_ai    3.1    4.9   4.6   4.7   5.8

### Figure 2 – Whole-process automation vs augmentation share, by platform (Releases 2–5)

In [3]:
collab = pd.concat([rc.compute_collab_mode_by_subprocess(tm, panel, p, (2, 3, 4, 5)) for p in rc.PLATFORMS],
                   ignore_index=True)
rc.save_csv(collab, "rq1", "fig02_04_data.csv")
whole = collab[collab["subprocess"] == "Entire Process"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, metric, title in zip(axes, ["automation_pct", "augmentation_pct"],
                             ["Automation share (usage-weighted)", "Augmentation share (usage-weighted)"]):
    for p in ["claude_ai", "1p_api"]:
        s = whole[whole["platform"] == p].sort_values("release")
        ax.plot(s["release"], s[metric], color=rc.PLATFORM_COLORS[p], marker=rc.PLATFORM_MARKERS[p],
                linewidth=2.4, markersize=6.5, markeredgecolor="white", markeredgewidth=0.8)
    ax.set_title(title, fontsize=11.5)
    rc.release_ticks(ax, [2, 3, 4, 5])
    ax.set_ylabel("% of conversations")
    rc.capped_ylim(ax, 100)
    rc.clean_axes(ax)
rc.legend_box(axes[0], [(rc.PLATFORM_COLORS[p], rc.PLATFORM_LABELS[p], rc.PLATFORM_MARKERS[p]) for p in ["claude_ai", "1p_api"]],
              loc="upper left")
fig.suptitle("Whole-process automation vs. augmentation share over time, by platform", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.94])
rc.save_figure(fig, "rq1", "fig02_collaboration_mode_by_platform.png")
whole.pivot_table(index="platform", columns="release", values=["automation_pct", "augmentation_pct"]).round(1)

[write] outputs/2_ch4_1_rq1_location/fig02_04_data.csv


[write] outputs/2_ch4_1_rq1_location/fig02_collaboration_mode_by_platform.png


augmentation_pct                   automation_pct                  
release                  2     3     4     5              2     3     4     5
platform                                                                     
1p_api                 NaN   5.8   9.3  11.2            NaN  86.8  83.5  77.4
claude_ai             50.5  48.7  51.1  54.5           36.1  39.9  37.5  35.4

### Figures 3 and 4 – Automation vs augmentation share by subprocess (Claude.ai R2–R5; 1P API R3–R5)

In [4]:
def collab_mode_trend(platform, releases, filename):
    df = collab[collab["platform"] == platform]
    fig, axd = plt.subplot_mosaic(mosaic, figsize=(17, 7.5))
    for key, sp in [("ALL", "Entire Process")] + list(sp_key.items()):
        ax = axd[key]
        s = df[df["subprocess"] == sp].sort_values("release")
        big = key == "ALL"
        for mode in ["automation", "augmentation"]:
            ax.plot(s["release"], s[f"{mode}_pct"], color=rc.COLLAB_MODE_COLORS[mode], marker=rc.COLLAB_MODE_MARKERS[mode],
                    linewidth=2.6 if big else 1.8, markersize=6.5 if big else 4.5,
                    markeredgecolor="white", markeredgewidth=0.8 if big else 0.6)
        ax.set_title(sp, fontsize=13 if big else 9.5)
        rc.release_ticks(ax, releases, short=not big)
        ax.tick_params(labelsize=9.5 if big else 7.5)
        ax.set_ylabel("Share of conversations (%)" if big else "Share (%)", fontsize=11 if big else 8,
                      labelpad=14 if big else 6)
        rc.capped_ylim(ax, 100)
        rc.clean_axes(ax)
    rc.legend_box(fig, [(rc.COLLAB_MODE_COLORS[m], m.capitalize(), rc.COLLAB_MODE_MARKERS[m]) for m in ["automation", "augmentation"]],
                  loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=2)
    fig.suptitle(f"Automation vs. augmentation share over time, by subprocess -- {rc.PLATFORM_LABELS[platform]} "
                 f"(R{releases[0]}-R{releases[-1]})", fontsize=13, fontweight="bold")
    fig.tight_layout(rect=[0, 0.04, 1, 0.94])
    rc.save_figure(fig, "rq1", filename)


collab_mode_trend("claude_ai", [2, 3, 4, 5], "fig03_collaboration_mode_by_subprocess_claude_ai.png")
collab_mode_trend("1p_api", [3, 4, 5], "fig04_collaboration_mode_by_subprocess_1p_api.png")
collab[collab["subprocess"] != "Entire Process"].pivot_table(
    index=["platform", "subprocess"], columns="release", values=["automation_pct", "augmentation_pct"]).round(1)

[write] outputs/2_ch4_1_rq1_location/fig03_collaboration_mode_by_subprocess_claude_ai.png


[write] outputs/2_ch4_1_rq1_location/fig04_collaboration_mode_by_subprocess_1p_api.png


augmentation_pct                   automation_pct                  
release                                              2     3     4     5              2     3     4     5
platform  subprocess                                                                                     
1p_api    Analysis & Business Insight              NaN   8.3  10.8  19.2            NaN  87.2  85.3  73.3
          Close & Consolidation                    NaN   0.0  48.3  27.7            NaN  76.0  41.7  56.6
          Master Data Management                   NaN   0.0   8.1   4.4            NaN  86.9  79.0  74.3
          Planning & Budgeting                     NaN   8.6   0.0  12.4            NaN  70.3  69.3  71.7
          Record                                   NaN   2.9   5.0   3.0            NaN  89.5  90.3  85.5
          Report                                   NaN   0.0   0.0   8.8            NaN  87.1  89.7  78.4
claude_ai Analysis & Business Insight             58.5  57.8  60.0  63.5           33.2  37.3  35.5  33.1
          Close & Consolidation                   46.5  36.0  54.9  70.9            0.0   0.0  32.9  20.5
          Master Data Management                  52.5  30.5  42.1  41.1           41.3  48.7  41.1  40.5
          Planning & Budgeting                    55.8  45.9  46.6  49.9           35.1  34.5  30.4  32.1
          Record                                  33.6  35.9  36.3  43.3           36.3  48.5  43.7  39.6
          Report                                  31.3  24.9  39.8  40.6           37.1  60.8  49.2  47.6

### Figure 5 – Automation overrepresentation index by subprocess (left Claude.ai, right 1P API)

In [5]:
LABEL_POS = {
    "claude_ai": {"Report": ("bottom", 4), "Master Data Management": ("bottom", 4), "Record": ("top", -6),
                  "Analysis & Business Insight": ("bottom", 4), "Planning & Budgeting": ("top", -6),
                  "Close & Consolidation": ("top", -6)},
    "1p_api": {"Record": ("bottom", 6), "Report": ("bottom", 3), "Master Data Management": ("bottom", 3),
               "Analysis & Business Insight": ("top", -6), "Planning & Budgeting": ("top", -9),
               "Close & Consolidation": ("bottom", 6)},
}
auto_idx = []
for p, fname in [("claude_ai", "fig05a_automation_index_claude_ai.png"), ("1p_api", "fig05b_automation_index_1p_api.png")]:
    df = rc.compute_automation_overrepresentation(tm, panel, p, (2, 3, 4, 5))
    auto_idx.append(df)
    rels = sorted(df["release"].unique())
    fig, ax = plt.subplots(figsize=(10.5, 6.5))
    ax.set_ylim(0, 1.6)
    last = {}
    for sp in rc.SUBPROCESS_ORDER:
        s = df[df["subprocess"] == sp].sort_values("release")
        if s.empty:
            continue
        ax.plot(s["release"], s["automation_index"], color=rc.SUBPROCESS_COLORS[sp], marker="o", linewidth=2.4,
                markersize=5.5, zorder=3, markeredgecolor="white", markeredgewidth=0.8)
        hl = s[s["release"] == 5]
        if not hl.empty and np.isfinite(hl["automation_index"].iloc[0]):
            v = float(hl["automation_index"].iloc[0])
            last[sp] = v
            va, dy = LABEL_POS[p].get(sp, ("bottom", 6))
            ax.annotate(f"{v:.1f}x", (5, v), xytext=(0, dy), textcoords="offset points", ha="center", va=va,
                        color=rc.SUBPROCESS_COLORS[sp], fontsize=9.5, fontweight="bold", zorder=6, path_effects=rc.TEXT_HALO)
    rc.end_of_line_labels(ax, last, rc.SUBPROCESS_COLORS, x_pos=max(rels) + 0.03, min_gap_frac=0.03,
                          min_gap_abs=0.035 if p == "1p_api" else 0.05, fontsize=9.5, clamp=True, dx=6)
    rc.baseline_hline(ax, 1.0)
    rc.release_ticks(ax, rels)
    ax.set_xlim(min(rels) - 0.15, max(rels) + 0.15)
    ax.set_ylabel("Automation overrepresentation index\n(subprocess automation % ÷ process automation %)")
    rc.clean_axes(ax)
    ax.set_title(f"Automation overrepresentation by subprocess — {rc.PLATFORM_LABELS[p]}", fontsize=12, loc="left")
    plt.subplots_adjust(right=0.72)
    rc.save_figure(fig, "rq1", fname)
pd.concat(auto_idx).pivot_table(index=["platform", "subprocess"], columns="release", values="automation_index").round(2)

findfont: Failed to find font weight semibold for Arial, now using 700.


[write] outputs/2_ch4_1_rq1_location/fig05a_automation_index_claude_ai.png


[write] outputs/2_ch4_1_rq1_location/fig05b_automation_index_1p_api.png


release                                   2     3     4     5
platform  subprocess                                         
1p_api    Analysis & Business Insight   NaN  1.00  1.02  0.95
          Close & Consolidation         NaN  0.88  0.50  0.73
          Master Data Management        NaN  1.00  0.95  0.96
          Planning & Budgeting          NaN  0.81  0.83  0.93
          Record                        NaN  1.03  1.08  1.10
          Report                        NaN  1.00  1.07  1.01
claude_ai Analysis & Business Insight  0.92  0.93  0.95  0.93
          Close & Consolidation        0.00  0.00  0.88  0.58
          Master Data Management       1.14  1.22  1.10  1.14
          Planning & Budgeting         0.97  0.86  0.81  0.91
          Record                       1.00  1.22  1.17  1.12
          Report                       1.03  1.53  1.31  1.34

## 4.1.2 Observed exposure by subprocess

### Figure 6 (R5) and Figure D1 (R4) – Theoretical capability vs observed exposure (radar)

In [6]:
def plot_radar(release, gate_on, section, filename):
    df = sub_exp[(sub_exp["release"] == release) & (sub_exp["gate_on"] == gate_on)].set_index("subprocess").reindex(rc.SUBPROCESS_ORDER)
    theoretical = df["theoretical_capability_share"].fillna(0).tolist()
    observed = df["R_o"].fillna(0).tolist()
    n = len(rc.SUBPROCESS_ORDER)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    close = lambda v: v + v[:1]
    BLUE, BLUE_FILL, RED, RED_FILL = "#2A84DC", "#B7D3EB", "#E23C3B", "#EF9595"
    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_facecolor("#E5ECF0")
    ax.set_xticks(angles)
    ax.set_xticklabels(["\n".join(textwrap.wrap(c, width=14)) for c in rc.SUBPROCESS_ORDER], fontsize=11, color="#595850")
    ax.tick_params(axis="x", pad=32)
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels([])
    ax.grid(color="#E6E4DD", linewidth=0.9)
    ax.spines["polar"].set_color("#A3A29D")
    ax.spines["polar"].set_linewidth(1.2)
    ax.plot(close(angles), close(theoretical), color=BLUE, linewidth=1.8, zorder=3, label="Theoretical Capability")
    ax.fill(close(angles), close(theoretical), color=BLUE_FILL, alpha=1.0, zorder=1)
    ax.plot(close(angles), close(observed), color=RED, linewidth=1.8, zorder=4, label="Observed Exposure")
    ax.fill(close(angles), close(observed), color=RED_FILL, alpha=1.0, zorder=2.5)
    for vals, col, z in [(theoretical, BLUE, 6), (observed, RED, 7)]:
        for a, v in zip(angles, vals):
            ax.annotate(f"{v:.2f}", xy=(a, v), ha="center", va="center", fontsize=8.5, fontweight="bold",
                        color="white", zorder=z, bbox=dict(fc=col, ec="none", boxstyle="square,pad=0.3"))
    fig.suptitle(f"Theoretical Capability and Observed Exposure by Subprocess (Release {release}, "
                 f"gate={'ON' if gate_on else 'OFF'})", fontsize=14, fontweight="bold", y=0.98)
    leg = ax.legend(loc="lower right", bbox_to_anchor=(1.18, -0.05), frameon=True, fontsize=10, borderpad=0.8)
    leg.get_frame().set_edgecolor("#CCCCCC")
    fig.subplots_adjust(top=0.90, bottom=0.05)
    rc.save_figure(fig, section, filename)


plot_radar(5, True, "rq1", "fig06a_radar_R5_gateON.png")
plot_radar(5, False, "rq1", "fig06b_radar_R5_gateOFF.png")
plot_radar(4, True, "appD", "figD1a_radar_R4_gateON.png")
plot_radar(4, False, "appD", "figD1b_radar_R4_gateOFF.png")
sub_exp.pivot_table(index="subprocess", columns=["release", "gate_on"], values=["theoretical_capability_share", "R_o"]).reindex(rc.SUBPROCESS_ORDER).round(2)

[write] outputs/2_ch4_1_rq1_location/fig06a_radar_R5_gateON.png


[write] outputs/2_ch4_1_rq1_location/fig06b_radar_R5_gateOFF.png


[write] outputs/4_appendix_D/figD1a_radar_R4_gateON.png


[write] outputs/4_appendix_D/figD1b_radar_R4_gateOFF.png


R_o                   theoretical_capability_share                  
release                         4           5                                  4           5      
gate_on                     False True  False True                         False True  False True 
subprocess                                                                                        
Master Data Management       0.51  0.25  0.56  0.25                         1.00  1.00  1.00  1.00
Record                       0.40  0.21  0.44  0.32                         0.97  0.97  0.97  0.97
Close & Consolidation        0.44  0.44  0.40  0.40                         1.00  1.00  1.00  1.00
Report                       0.15  0.06  0.21  0.06                         1.00  1.00  1.00  1.00
Analysis & Business Insight  0.47  0.18  0.51  0.18                         1.00  1.00  1.00  1.00
Planning & Budgeting         0.20  0.09  0.16  0.09                         1.00  1.00  1.00  1.00

## 4.1.3 Task-level concentration of exposure

### Figure 7 – Subprocess concentration vs subprocess exposure (R5; left gate ON, right gate OFF)
Axes are shared across R4/R5 × gate so the two published panels are directly comparable. The blue square is the entire process (Rₒ,ₚ, process concentration); it is excluded from the OLS fit.

In [7]:
panels = [(r, g) for r in rc.EXPOSURE_RELEASES for g in rc.GATE_STATES]
_, ylim = rc.global_limits([sub_conc[(sub_conc["release"] == r) & (sub_conc["gate_on"] == g)] for r, g in panels], "R_o", "concentration")
ylim = (ylim[0], max(ylim[1], proc_conc["concentration"].max() * 1.15))
xlim = (0.0, 1.0)
for g, letter in [(True, "a"), (False, "b")]:
    d = sub_conc[(sub_conc["release"] == 5) & (sub_conc["gate_on"] == g)]
    pc = proc_conc[(proc_conc["release"] == 5) & (proc_conc["gate_on"] == g)].iloc[0]
    fig, ax = plt.subplots(figsize=(8, 6.5))
    rc.clean_axes(ax)
    rc.add_quadrant_lines(ax, xlim, ylim)
    rc.subprocess_bubbles(ax, d, "R_o", "concentration")
    rc.draw_entire_process(ax, pc["R_process"], pc["concentration"])
    slope, icpt, r2, n, lab = rc.trend_line(ax, d["R_o"], d["concentration"], xlim)
    rc.record_fit("Figure 7", f"R5 {gate_txt(g)}", "R_o", "concentration", slope, icpt, r2, n)
    rc.legend_box(ax, [(rc.MODE_COLOR_COOL, lab, "line")], loc="upper right")
    ax.set_xlabel("Subprocess exposure (R_o)")
    ax.set_ylabel("Subprocess concentration")
    ax.set_title(f"Subprocess concentration vs exposure -- Release 5, {gate_txt(g)}")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    rc.save_figure(fig, "rq1", f"fig07{letter}_concentration_vs_exposure_R5_{gate_tag(g)}.png")
print("Concentration median (R5):", sub_conc[sub_conc["release"] == 5].groupby("gate_on")["concentration"].median().round(3).to_dict())

findfont: Failed to find font weight semibold for Arial, now using 700.


findfont: Failed to find font weight semibold for Arial, now using 700.


[write] outputs/2_ch4_1_rq1_location/fig07a_concentration_vs_exposure_R5_gateON.png


[write] outputs/2_ch4_1_rq1_location/fig07b_concentration_vs_exposure_R5_gateOFF.png
Concentration median (R5): {False: 0.986, True: 0.982}


### Figure 8 – Concentration by subprocess and entire process, Release 4 → 5 (left gate ON, right gate OFF)

In [8]:
y_all = pd.concat([sub_conc["concentration"], proc_conc["concentration"]]).dropna()
ylim8 = (min(0, y_all.min() - (y_all.max() - y_all.min()) * 0.12), y_all.max() + (y_all.max() - y_all.min()) * 0.12)
for g, letter in [(True, "a"), (False, "b")]:
    fig, ax = plt.subplots(figsize=(9, 6.5))
    rc.clean_axes(ax, grid="y")
    last, colors = {}, {}
    for sp in rc.SUBPROCESS_ORDER:
        s = sub_conc[(sub_conc["gate_on"] == g) & (sub_conc["subprocess"] == sp)].sort_values("release")
        ax.plot(s["release"], s["concentration"], color=rc.SUBPROCESS_COLORS[sp], marker="o", linewidth=2.0,
                markersize=6, alpha=0.9, zorder=3)
        last[sp], colors[sp] = s["concentration"].iloc[-1], rc.SUBPROCESS_COLORS[sp]
    s = proc_conc[proc_conc["gate_on"] == g].sort_values("release")
    ax.plot(s["release"], s["concentration"], color=rc.ENTIRE_PROCESS_COLOR, marker="s", markersize=6, linewidth=2.0, zorder=5)
    last["Entire Process"], colors["Entire Process"] = s["concentration"].iloc[-1], rc.ENTIRE_PROCESS_COLOR
    rc.end_of_line_labels(ax, last, colors, x_pos=5, min_gap_frac=0.035, min_gap_abs=0.06)
    ax.set_xticks([4, 5])
    ax.set_xticklabels(["Release 4", "Release 5"])
    ax.set_xlim(3.85, 5.55)
    ax.set_ylim(*ylim8)
    ax.set_ylabel("Concentration score")
    ax.set_title(f"Concentration by subprocess and entire process -- {gate_txt(g)}")
    fig.subplots_adjust(right=0.78)
    rc.save_figure(fig, "rq1", f"fig08{letter}_concentration_trend_{gate_tag(g)}.png")
pd.concat([sub_conc.pivot_table(index="subprocess", columns=["gate_on", "release"], values="concentration").reindex(rc.SUBPROCESS_ORDER),
           proc_conc.pivot_table(columns=["gate_on", "release"], values="concentration").rename(index={"concentration": "Entire Process"})]).round(2)

findfont: Failed to find font weight semibold for Arial, now using 700.


[write] outputs/2_ch4_1_rq1_location/fig08a_concentration_trend_gateON.png


[write] outputs/2_ch4_1_rq1_location/fig08b_concentration_trend_gateOFF.png


gate_on                     False       True       
release                         4     5     4     5
Master Data Management       1.13  1.04  1.09  1.10
Record                       1.58  1.38  1.09  1.46
Close & Consolidation        1.13  0.93  1.13  0.93
Report                       0.72  0.74  0.31  0.31
Analysis & Business Insight  1.54  1.42  1.03  1.03
Planning & Budgeting         0.66  0.49  0.35  0.37
Entire Process               1.60  1.47  1.03  1.21

### Figure 9 – Task dispersion contribution vs subprocess exposure (R5; left gate ON, right gate OFF)
Each task sits at its parent subprocess's Rₒ; the six highest-dispersion tasks are labelled with their O*NET task ID.

In [9]:
dfs = [task_disp[(task_disp["release"] == r) & (task_disp["gate_on"] == g)] for r, g in panels]
_, ylim9 = rc.global_limits(dfs, "subprocess_R_o", "subprocess_weighted_dispersion", 0.08, 0.12)
for g, letter in [(True, "a"), (False, "b")]:
    d = task_disp[(task_disp["release"] == 5) & (task_disp["gate_on"] == g)].dropna(subset=["subprocess_R_o", "subprocess_weighted_dispersion"])
    fig, ax = plt.subplots(figsize=(9, 7))
    rc.clean_axes(ax)
    rc.add_quadrant_lines(ax, xlim, ylim9)
    rc.task_bubbles(ax, d, "subprocess_R_o", "subprocess_weighted_dispersion")
    slope, icpt, r2, n, lab = rc.trend_line(ax, d["subprocess_R_o"], d["subprocess_weighted_dispersion"], xlim)
    rc.record_fit("Figure 9", f"R5 {gate_txt(g)}", "subprocess R_o", "task dispersion", slope, icpt, r2, n)
    rc.legend_box(ax, [(rc.MODE_COLOR_COOL, lab, "line")] + [(rc.SUBPROCESS_COLORS[sp], sp, "o") for sp in rc.SUBPROCESS_ORDER],
                  loc="upper right", ncol=1)
    ax.set_xlabel("Subprocess exposure (R_o)")
    ax.set_ylabel("Task dispersion contribution (subprocess-level)")
    ax.set_title(f"Task dispersion vs subprocess exposure -- Release 5, {gate_txt(g)}")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim9)
    rc.save_figure(fig, "rq1", f"fig09{letter}_task_dispersion_vs_exposure_R5_{gate_tag(g)}.png")
    print(f"Top dispersion tasks, R5 {gate_txt(g)}:")
    print(d.nlargest(6, "subprocess_weighted_dispersion")[["task_id", "subprocess", "r_tilde", "WorkUsage", "subprocess_weighted_dispersion"]].round(4).to_string(index=False))

[write] outputs/2_ch4_1_rq1_location/fig09a_task_dispersion_vs_exposure_R5_gateON.png
Top dispersion tasks, R5 gate ON:
task_id                  subprocess  r_tilde  WorkUsage  subprocess_weighted_dispersion
  21321 Analysis & Business Insight   0.8893      995.0                          0.0664
   1305      Master Data Management   0.8736      161.0                          0.0563
  23243       Close & Consolidation   0.8130      206.0                          0.0487
   2492                      Report   0.8884      464.0                          0.0453
  21516       Close & Consolidation   0.0000        0.0                          0.0428
  21521      Master Data Management   0.8753      848.0                          0.0378


[write] outputs/2_ch4_1_rq1_location/fig09b_task_dispersion_vs_exposure_R5_gateOFF.png
Top dispersion tasks, R5 gate OFF:
task_id                  subprocess  r_tilde  WorkUsage  subprocess_weighted_dispersion
   1306      Master Data Management   0.0000        0.0                          0.0574
  23243       Close & Consolidation   0.8130      206.0                          0.0487
   8856                      Report   0.8686       79.0                          0.0438
  21516       Close & Consolidation   0.0000        0.0                          0.0428
     89      Master Data Management   0.0000        0.0                          0.0362
   3394 Analysis & Business Insight   0.0000        0.0                          0.0331


### Figure 10 – Lorenz curves (R5, left) and Gini coefficient by platform across releases (right)

In [10]:
def lorenz(values):
    v = np.sort(np.asarray(values, dtype=float)[np.asarray(values, dtype=float) > 0])
    if len(v) == 0:
        return np.array([0, 100]), np.array([0, 100]), np.nan
    n = len(v)
    gini = (2 * np.sum(np.arange(1, n + 1) * v)) / (n * v.sum()) - (n + 1) / n
    x = np.insert(np.arange(1, n + 1) / n, 0, 0) * 100
    y = np.insert(np.cumsum(v) / v.sum(), 0, 0) * 100
    return x, y, gini


gini_rows = []
for r in rc.ANALYSIS_RELEASES:
    for p in rc.RELEASE_PLATFORMS[r]:
        x, y, gini = lorenz(rc.get_r2r_task_pct(panel, r, p).values)
        gini_rows.append({"release": r, "platform": p, "gini": gini, "top20_share_pct": 100 - float(np.interp(80, x, y))})
gini_df = pd.DataFrame(gini_rows)
rc.save_csv(gini_df, "rq1", "fig10_gini_top20.csv")

fig, ax = plt.subplots(figsize=(7.8, 7.8))
ax.plot([0, 100], [0, 100], linestyle="--", color=rc.GRID_COLOR, linewidth=1.3, zorder=1)
entries, notes = [(rc.GRID_COLOR, "Perfect equality", "line")], []
for p in ["1p_api", "claude_ai"]:
    x, y, gini = lorenz(rc.get_r2r_task_pct(panel, 5, p).values)
    ax.plot(x, y, color=rc.PLATFORM_COLORS[p], linewidth=2.4, zorder=2)
    entries.append((rc.PLATFORM_COLORS[p], f"{rc.PLATFORM_LABELS[p]} (Gini = {gini:.3f})", rc.PLATFORM_MARKERS[p]))
    y80 = float(np.interp(80, x, y))
    ax.scatter([80], [y80], color=rc.PLATFORM_COLORS[p], s=45, zorder=4, edgecolor="white", linewidth=0.8)
    dx, dy, ha, va = (-6, 8, "right", "bottom") if p == "1p_api" else (6, -8, "left", "top")
    ax.annotate(f"{y80:.1f}%", (80, y80), xytext=(dx, dy), textcoords="offset points", ha=ha, va=va,
                color=rc.PLATFORM_COLORS[p], fontsize=10, fontweight="bold", zorder=5)
    notes.append(f"Top 20% of {rc.PLATFORM_LABELS[p]} tasks = {100 - y80:.1f}% of usage")
ax.axvline(80, color=rc.GRID_COLOR, linewidth=rc.GRID_LINEWIDTH, zorder=1)
ax.set_xlabel("Cumulative percentage of tasks")
ax.set_ylabel("Cumulative percentage of usage")
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
rc.clean_axes(ax)
rc.legend_box(ax, entries, loc="upper left", ncol=1)
ax.text(0.02, 0.87, "\n".join(notes), transform=ax.transAxes, fontsize=9.5, va="top", ha="left",
        bbox=dict(boxstyle="round,pad=0.5", fc=rc.LEGEND_FACE_COLOR, ec=rc.LEGEND_EDGE_COLOR, linewidth=1))
fig.tight_layout()
rc.save_figure(fig, "rq1", "fig10a_lorenz_R5.png")

force_below = {("claude_ai", 3), ("1p_api", 4), ("1p_api", 5)}
fig, ax = plt.subplots(figsize=(9, 6.5))
for p in ["claude_ai", "1p_api"]:
    s = gini_df[gini_df["platform"] == p].sort_values("release")
    ax.plot(s["release"], s["gini"], color=rc.PLATFORM_COLORS[p], marker=rc.PLATFORM_MARKERS[p], linewidth=2.4,
            markersize=6.5, markeredgecolor="white", markeredgewidth=0.8)
    for _, row in s.iterrows():
        below = (p, int(row["release"])) in force_below
        ax.annotate(f"{row['gini']:.3f}", (row["release"], row["gini"]), xytext=(0, -14 if below else 10),
                    textcoords="offset points", ha="center", va="top" if below else "bottom",
                    color=rc.PLATFORM_COLORS[p], fontsize=9.5, fontweight="semibold")
rc.release_ticks(ax, rc.ANALYSIS_RELEASES)
ax.set_ylabel("Gini coefficient")
ax.set_ylim(0, 1)
rc.clean_axes(ax)
rc.legend_box(ax, [(rc.PLATFORM_COLORS[p], rc.PLATFORM_LABELS[p], rc.PLATFORM_MARKERS[p]) for p in ["claude_ai", "1p_api"]], loc="lower right")
fig.tight_layout()
rc.save_figure(fig, "rq1", "fig10b_gini_trend.png")
gini_df.round(3)

[write] outputs/2_ch4_1_rq1_location/fig10_gini_top20.csv


[write] outputs/2_ch4_1_rq1_location/fig10a_lorenz_R5.png


[write] outputs/2_ch4_1_rq1_location/fig10b_gini_trend.png


,release,platform,gini,top20_share_pct
0,1,claude_ai,0.718,78.759
1,2,claude_ai,0.649,71.242
2,3,claude_ai,0.671,72.102
3,3,1p_api,0.726,80.261
4,4,claude_ai,0.714,77.065
5,4,1p_api,0.660,72.918
6,5,claude_ai,0.713,76.351
7,5,1p_api,0.663,72.611


## Fit statistics quoted in Section 4.1

In [11]:
rc.write_fit_log("rq1")

[write] outputs/2_ch4_1_rq1_location/fit_statistics.csv


,figure,panel,x,y,slope,intercept,r2,n,note
0,Figure 7,R5 gate ON,R_o,concentration,2.549522,0.313983,0.582029,6,
1,Figure 7,R5 gate OFF,R_o,concentration,1.861469,0.290094,0.683499,6,
2,Figure 9,R5 gate ON,subprocess R_o,task dispersion,0.035344,0.000674,0.092108,91,
3,Figure 9,R5 gate OFF,subprocess R_o,task dispersion,0.022633,0.001070,0.082990,91,
